In [3]:
import re
import subprocess
from pathlib import Path

# Keep this block minimal and close to the original notebook style.
GAME                 = "antichess"
GENERATED_CODE_PATH  = f"outputs/{GAME}.py"

LLM_MODEL            = "openai-codex/gpt-5.5:xhigh"
TIMEOUT_SECONDS      = 600
OPEN_SPIEL_MAX_STEPS = 40
LEGAL_ACTION_LIMIT   = 8
COMPARE_SEED         = None
PREFERRED_MOVES      = []

EXTRA_CONTEXT_PATHS        = [Path("prompts/open_spiel_backbone.md")]
IMPLEMENTATION_BRIEF_PATH  = Path(f"outputs/{GAME}_implementation_brief.md")
JUDGE_REVIEW_PATH          = Path(f"outputs/{GAME}_judge.md")
CODE_PATH            = Path(GENERATED_CODE_PATH)
RESPONSE_PATH        = CODE_PATH.with_suffix(".md")


## LLM implementation

Generates code from the rulebook, main prompt, and optional backbone/brief context.


In [ ]:
try:
    import shutil

    if not LLM_MODEL:
        raise ValueError("Set LLM_MODEL")

    def resolve_input_dir():
        candidates = [Path("inputs")]
        for candidate in candidates:
            if candidate.exists():
                return candidate
        return candidates[0]

    def resolve_prompt_path():
        candidates = [Path("prompts/rulebook_to_python.txt")]
        for candidate in candidates:
            if candidate.exists():
                return candidate
        raise FileNotFoundError("Missing prompt file: prompts/rulebook_to_python.txt")

    def find_rules_path(input_dir):
        if not input_dir.exists():
            raise FileNotFoundError(f"Missing input directory: {input_dir}")

        rules_paths = sorted(
            path
            for path in input_dir.iterdir()
            if path.stem == "game_rules" and path.suffix.lower() in {".txt", ".pdf"}
        )
        if not rules_paths:
            raise FileNotFoundError(f"Missing {input_dir / 'game_rules.txt'} or {input_dir / 'game_rules.pdf'}")
        if len(rules_paths) > 1:
            names = ", ".join(path.name for path in rules_paths)
            raise RuntimeError(f"Multiple game_rules files found; keep exactly one: {names}")
        return rules_paths[0]

    def read_rules_text(rules_path):
        if rules_path.suffix.lower() == ".txt":
            return rules_path.read_text(encoding="utf-8")

        if rules_path.suffix.lower() == ".pdf":
            try:
                from pypdf import PdfReader
            except ImportError as exc:
                raise ImportError("PDF rulebooks require pypdf; install requirements.txt") from exc

            reader = PdfReader(str(rules_path))
            text = "\n\n".join(
                page_text.strip()
                for page in reader.pages
                for page_text in [page.extract_text() or ""]
                if page_text.strip()
            )
            if not text:
                raise ValueError(f"No extractable text found in {rules_path}")
            return text

        raise ValueError(f"Unsupported rules file type: {rules_path.suffix}")

    input_dir = resolve_input_dir()
    prompt_path = resolve_prompt_path()
    prompt_text = prompt_path.read_text(encoding="utf-8")
    rules_path = find_rules_path(input_dir)
    rules_text = read_rules_text(rules_path)
    print(f"Using prompt file: {prompt_path}")
    print(f"Using rules file: {rules_path}")

    prompt_parts = [prompt_text]
    for extra_path in EXTRA_CONTEXT_PATHS:
        if extra_path.exists():
            print(f"Using extra context: {extra_path}")
            prompt_parts.append(f"# Extra context: {extra_path}\n\n" + extra_path.read_text(encoding="utf-8"))
    if IMPLEMENTATION_BRIEF_PATH.exists():
        print(f"Using implementation brief: {IMPLEMENTATION_BRIEF_PATH}")
        prompt_parts.append(f"# Implementation brief: {IMPLEMENTATION_BRIEF_PATH}\n\n" + IMPLEMENTATION_BRIEF_PATH.read_text(encoding="utf-8"))

    full_prompt = "\n\n".join(prompt_parts) + "\n\nHier folgt die Spielanleitung:\n\n" + rules_text

    pi_path = shutil.which("pi") or shutil.which("pi.cmd")
    if pi_path is None:
        fallback = Path.home() / "AppData/Roaming/npm/pi.cmd"
        if not fallback.exists():
            raise FileNotFoundError("Could not find pi or pi.cmd")
        pi_path = str(fallback)

    result = subprocess.run(
        [pi_path, "-p", "--model", LLM_MODEL],
        input=full_prompt,
        capture_output=True,
        text=True,
        timeout=TIMEOUT_SECONDS,
    )

    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip() or "pi call failed")

    CODE_PATH.parent.mkdir(parents=True, exist_ok=True)
    RESPONSE_PATH.write_text(result.stdout, encoding="utf-8")

    match = re.search(r"```python\s*(.*?)```", result.stdout, re.IGNORECASE | re.DOTALL)
    if match is None:
        raise RuntimeError("No fenced python block found in the LLM response")

    CODE_PATH.write_text(match.group(1).strip() + "\n", encoding="utf-8")
    print(f"Saved raw response: {RESPONSE_PATH}")
    print(f"Saved extracted code: {CODE_PATH}")
except Exception as exc:
    print(f"LLM call failed: {exc}")



## Checks

Run after generation and before manual inspection.

- normal checks run by default
- set `INCLUDE_LLM_JUDGE=True` after saving a judge review
- set `INCLUDE_OPENSPIEL_COMPARE=True` for the optional OpenSpiel comparison


In [4]:
import subprocess
import sys
from pathlib import Path

# Toggle these here when rerunning checks.
INCLUDE_LLM_JUDGE = False
INCLUDE_OPENSPIEL_COMPARE = False

check_script = next(
    (path for path in [Path("checks/run_checks.py"), Path("../checks/run_checks.py")] if path.exists()),
    None,
)
if check_script is None:
    raise FileNotFoundError("Could not find checks/run_checks.py")

cmd = [
    sys.executable,
    "-u",
    str(check_script),
    "--game",
    GAME,
    "--code-path",
    str(CODE_PATH),
    "--rollouts",
    "1000",
]
if INCLUDE_LLM_JUDGE:
    cmd.append("--include-judge")
    if JUDGE_REVIEW_PATH:
        cmd += ["--judge-path", str(JUDGE_REVIEW_PATH)]
if INCLUDE_OPENSPIEL_COMPARE:
    cmd.append("--include-final")

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

if process.stdout is None:
    raise RuntimeError("could not read check output")

for line in process.stdout:
    if line.startswith(("OK", "FAIL", "summary:")):
        print(line, end="", flush=True)

returncode = process.wait()
if returncode != 0:
    print(f"checks failed with exit code {returncode}")


OK   01_result_file (1/1, 0.00s)
OK   02_python_syntax (1/1, 0.00s)
OK   03_startable_game (1/1, 0.00s)
OK   04_required_api (8/8, 0.00s)
OK   05_random_rollouts (1000/1000, 28.20s)
summary: 5/5 checks, 1011/1011 units, 28.21s


## Optional manual inspection

Collapsed and skipped by default. Normal workflow ends after `## Checks`.
Set `RUN_MANUAL_EXPERIMENTS = True` in the next hidden code cell only when you want ad-hoc loading/inspection.


In [2]:
RUN_MANUAL_EXPERIMENTS = False
if RUN_MANUAL_EXPERIMENTS:
    import importlib.util
    import sys
    import pyspiel

    try:
        game = pyspiel.load_game(GAME)

        if not CODE_PATH.exists():
            raise FileNotFoundError(f"Generated code missing: {CODE_PATH}")

        spec = importlib.util.spec_from_file_location(GAME, CODE_PATH)
        module = importlib.util.module_from_spec(spec)
        if spec is None or spec.loader is None:
            raise RuntimeError("Could not load generated game module")
        sys.modules[spec.name] = module
        spec.loader.exec_module(module)
        llm_game = module.Game()
    except Exception as exc:
        print(f"Game loading failed: {exc}")


## OpenSpiel basics: actions

OpenSpiel trennt zwischen `Game` und `State`. `pyspiel.load_game(GAME)` lädt die Referenzimplementierung; `game.new_initial_state()` erzeugt einen konkreten Startzustand. Aktionen sind intern Integer-IDs. Welche IDs gerade legal sind, hängt vom aktuellen Zustand und Spieler ab.

Wichtige Methoden:

- `state.current_player()` gibt den Spieler zurück, der am Zug ist.
- `state.legal_actions(player)` liefert die im aktuellen Zustand legalen Action-IDs.
- `state.action_to_string(player, action)` übersetzt eine Action-ID im Kontext dieses Zustands in eine lesbare Notation.
- `state.clone()` kopiert einen Zustand, damit man Testaktionen anwenden kann, ohne das Original zu verändern.
- `state.apply_action(action)` führt eine legale Aktion aus.
- `game.num_distinct_actions()` beschreibt den globalen Action-ID-Raum; in einem einzelnen Zustand ist meistens nur ein kleiner Teil davon legal.

Für Vergleiche sollte man deshalb nicht nur rohe Integer-IDs betrachten, sondern immer Zustand, Spieler, legale Actions und die lesbare Bedeutung zusammen prüfen.


In [4]:
if RUN_MANUAL_EXPERIMENTS:
    # Small OpenSpiel action inspection.
    # Run the loading cell above first so `game` exists.

    state = game.new_initial_state()
    player = state.current_player()
    legal_actions = state.legal_actions(player)

    print(f"game: {GAME}")
    print(f"players: {game.num_players()}")
    print(f"global action-id space: 0..{game.num_distinct_actions() - 1}")
    print(f"current player: {player}")
    print(f"terminal: {state.is_terminal()}")

    print("\ninitial state:")
    print(state)

    print(f"\nlegal actions for player {player}: {len(legal_actions)}")
    for index, action in enumerate(legal_actions, start=1):
        label = state.action_to_string(player, action)
        print(f"{index:>2}. action id {action:>5} -> {label}")

    if legal_actions:
        example_action = legal_actions[0]
        print("\nexample action_to_string call:")
        print(f"state.action_to_string({player}, {example_action}) = {state.action_to_string(player, example_action)!r}")

        next_state = state.clone()
        next_state.apply_action(example_action)
        print("\nafter applying the first legal action:")
        print(next_state)

    # Set to True only if you really want to print the full global action-id space.
    # For chess-like games this can be several thousand IDs, and many are not legal in this state.
    SHOW_FULL_ACTION_ID_SPACE = False

    if SHOW_FULL_ACTION_ID_SPACE:
        legal_action_set = set(legal_actions)
        print(f"\nall global action ids: 0..{game.num_distinct_actions() - 1}")
        for action in range(game.num_distinct_actions()):
            try:
                label = state.action_to_string(player, action)
            except Exception as exc:
                label = f"<not printable in this state: {exc}>"
            marker = "*" if action in legal_action_set else " "
            print(f"{marker} action id {action:>5} -> {label}")


game: antichess
players: 2
global action-id space: 0..4863
current player: 1
terminal: False

initial state:
rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w - - 0 1

legal actions for player 1: 20
 1. action id    95 -> a3
 2. action id    96 -> a4
 3. action id   679 -> Na3
 4. action id   683 -> Nc3
 5. action id   703 -> b3
 6. action id   704 -> b4
 7. action id  1311 -> c3
 8. action id  1312 -> c4
 9. action id  1919 -> d3
10. action id  1920 -> d4
11. action id  2527 -> e3
12. action id  2528 -> e4
13. action id  3135 -> f3
14. action id  3136 -> f4
15. action id  3719 -> Nf3
16. action id  3723 -> Nh3
17. action id  3743 -> g3
18. action id  3744 -> g4
19. action id  4351 -> h3
20. action id  4352 -> h4

example action_to_string call:
state.action_to_string(1, 95) = 'a3'

after applying the first legal action:
rnbqkbnr/pppppppp/8/8/8/P7/1PPPPPPP/RNBQKBNR b - - 0 1


In [ ]:
if RUN_MANUAL_EXPERIMENTS:
    # Same basic OpenSpiel action inspection for Mühle / Nine Men's Morris.
    # This uses separate variable names so the antichess comparison below is not changed.

    MUEHLE_GAME = "nine_mens_morris"
    MUEHLE_ACTION_PRINT_LIMIT = 40

    try:
        muehle_game = pyspiel.load_game(MUEHLE_GAME)
        muehle_state = muehle_game.new_initial_state()
        muehle_player = muehle_state.current_player()
        muehle_actions = muehle_state.legal_actions(muehle_player)

        print(f"game: {MUEHLE_GAME}")
        print(f"players: {muehle_game.num_players()}")
        print(f"global action-id space: 0..{muehle_game.num_distinct_actions() - 1}")
        print(f"current player: {muehle_player}")
        print(f"terminal: {muehle_state.is_terminal()}")

        print("\ninitial state:")
        print(muehle_state)

        print(f"\nlegal actions for player {muehle_player}: {len(muehle_actions)}")
        for index, action in enumerate(muehle_actions[:MUEHLE_ACTION_PRINT_LIMIT], start=1):
            label = muehle_state.action_to_string(muehle_player, action)
            print(f"{index:>2}. action id {action:>5} -> {label}")
        if len(muehle_actions) > MUEHLE_ACTION_PRINT_LIMIT:
            print(f"... {len(muehle_actions) - MUEHLE_ACTION_PRINT_LIMIT} more legal actions not shown")

        if muehle_actions:
            muehle_example_action = muehle_actions[0]
            print("\nexample action_to_string call:")
            print(
                f"muehle_state.action_to_string({muehle_player}, {muehle_example_action}) = "
                f"{muehle_state.action_to_string(muehle_player, muehle_example_action)!r}"
            )

            muehle_next_state = muehle_state.clone()
            muehle_next_state.apply_action(muehle_example_action)
            print("\nafter applying the first legal action:")
            print(muehle_next_state)

            if not muehle_next_state.is_terminal():
                muehle_next_player = muehle_next_state.current_player()
                muehle_next_actions = muehle_next_state.legal_actions(muehle_next_player)
                print(f"\nnext legal actions for player {muehle_next_player}: {len(muehle_next_actions)}")
                for index, action in enumerate(muehle_next_actions[:MUEHLE_ACTION_PRINT_LIMIT], start=1):
                    label = muehle_next_state.action_to_string(muehle_next_player, action)
                    print(f"{index:>2}. action id {action:>5} -> {label}")
                if len(muehle_next_actions) > MUEHLE_ACTION_PRINT_LIMIT:
                    print(f"... {len(muehle_next_actions) - MUEHLE_ACTION_PRINT_LIMIT} more legal actions not shown")
    except Exception as exc:
        print(f"Mühle action inspection failed: {exc}")


game: nine_mens_morris
players: 2
global action-id space: 0..599
current player: 0
terminal: False

initial state:
.------.------.
|      |      |
| .----.----. |
| |    |    | |
| | .--.--. | |
| | |     | | |
.-.-.     .-.-.
| | |     | | |
| | .--.--. | |
| |    |    | |
| .----.----. |
|      |      |
.------.------.

Current player: W
Turn number: 0
Men to deploy: 9 9
Num men: 9 9


legal actions for player 0: 24
 1. action id     0 -> Point 0
 2. action id     1 -> Point 1
 3. action id     2 -> Point 2
 4. action id     3 -> Point 3
 5. action id     4 -> Point 4
 6. action id     5 -> Point 5
 7. action id     6 -> Point 6
 8. action id     7 -> Point 7
 9. action id     8 -> Point 8
10. action id     9 -> Point 9
11. action id    10 -> Point 10
12. action id    11 -> Point 11
13. action id    12 -> Point 12
14. action id    13 -> Point 13
15. action id    14 -> Point 14
16. action id    15 -> Point 15
17. action id    16 -> Point 16
18. action id    17 -> Point 17
19. action i

## Seeded random comparison


In [21]:
if RUN_MANUAL_EXPERIMENTS:
    try:
        import random
        import time

        PIECE_MAP = {"B": "P", "S": "N", "L": "B", "T": "R", "D": "Q", "K": "K"}

        def side_by_side(left, right, width=56):
            left_lines = left.splitlines() or [""]
            right_lines = right.splitlines() or [""]
            height = max(len(left_lines), len(right_lines))
            left_lines += [""] * (height - len(left_lines))
            right_lines += [""] * (height - len(right_lines))
            return "\n".join(
                f"{left_line:<{width}} | {right_line}"
                for left_line, right_line in zip(left_lines, right_lines)
            )

        def open_visible_state(state):
            fen = state.to_string().split()
            board_part = fen[0]
            side = "white" if fen[1] == "w" else "black"

            board = [None] * 64
            for rank_i, row in enumerate(board_part.split("/")[::-1]):
                file_i = 0
                for char in row:
                    if char.isdigit():
                        file_i += int(char)
                    else:
                        board[rank_i * 8 + file_i] = ("w" if char.isupper() else "b") + char.upper()
                        file_i += 1

            return tuple(board), side

        def llm_visible_state(state):
            board = []
            for piece in state.board:
                if piece is None:
                    board.append(None)
                else:
                    board.append(piece[0] + PIECE_MAP[piece[1]])

            side = "white" if state.to_move == 0 else "black"
            return tuple(board), side

        def format_board(board):
            lines = ["   a  b  c  d  e  f  g  h"]
            for rank_i in range(7, -1, -1):
                row = []
                for file_i in range(8):
                    piece = board[rank_i * 8 + file_i]
                    row.append(piece if piece is not None else "..")
                lines.append(f"{rank_i + 1} " + " ".join(row))
            return "\n".join(lines)

        def panel(title, board, side):
            return f"{title}\nto move: {side}\n{format_board(board)}"

        def llm_apply(state, action):
            next_state = llm_game.apply_action(state, action)
            return state if next_state is None else next_state

        seed = COMPARE_SEED if COMPARE_SEED is not None else int(time.time() * 1000) % 1_000_000_000
        rng = random.Random(seed)

        print(f"compare seed: {seed}")
        print("reference: openspiel")
        print("state matching: visible board + side to move")

        open_state = game.new_initial_state()
        llm_state = llm_game.initial_state()
        history = []

        for step in range(OPEN_SPIEL_MAX_STEPS):
            open_board, open_side = open_visible_state(open_state)
            llm_board, llm_side = llm_visible_state(llm_state)
            open_terminal = open_state.is_terminal()
            llm_terminal = llm_game.is_terminal(llm_state)

            print("=" * 96)
            print(f"step {step}")
            if history:
                print("recent moves:", " ".join(history[-8:]))

            if (open_board, open_side) != (llm_board, llm_side):
                print("visible state mismatch")
                print(side_by_side(
                    panel("reference", open_board, open_side),
                    panel("generated", llm_board, llm_side),
                ))
                break

            if open_terminal != llm_terminal:
                print(f"terminal mismatch: reference={open_terminal} | generated={llm_terminal}")
                print(side_by_side(
                    panel("reference", open_board, open_side),
                    panel("generated", llm_board, llm_side),
                ))
                break

            print(side_by_side(
                panel("reference", open_board, open_side),
                panel("generated", llm_board, llm_side),
            ))

            if open_terminal:
                print(f"returns: reference={open_state.returns()} | generated={llm_game.returns(llm_state)}")
                break

            open_actions = list(open_state.legal_actions())
            llm_actions = list(llm_game.legal_actions(llm_state))
            matchable = []

            for open_action in open_actions:
                open_next = open_state.clone()
                open_next.apply_action(open_action)
                target = open_visible_state(open_next)

                llm_matches = []
                for llm_action in llm_actions:
                    if llm_visible_state(llm_apply(llm_state, llm_action)) == target:
                        llm_matches.append(llm_action)

                if llm_matches:
                    matchable.append((open_action, llm_matches))

            print(
                f"legal moves: reference={len(open_actions)} | generated={len(llm_actions)} | matching reference moves={len(matchable)}/{len(open_actions)}"
            )

            if len(matchable) != len(open_actions):
                matched_open = {
                    open_state.action_to_string(open_state.current_player(), open_action)
                    for open_action, _ in matchable
                }
                open_names = sorted(
                    open_state.action_to_string(open_state.current_player(), action)
                    for action in open_actions
                )
                print("unmatched reference sample:", [name for name in open_names if name not in matched_open][:LEGAL_ACTION_LIMIT])

            if not matchable:
                llm_names = sorted(llm_game.action_to_name(action) for action in llm_actions)
                print("generated sample:", llm_names[:LEGAL_ACTION_LIMIT])
                break

            if step < len(PREFERRED_MOVES):
                preferred = PREFERRED_MOVES[step]
                preferred_pairs = [
                    (open_action, llm_matches)
                    for open_action, llm_matches in matchable
                    if open_state.action_to_string(open_state.current_player(), open_action) == preferred
                ]
                if not preferred_pairs:
                    print(f"preferred move not matchable: {preferred}")
                    break
                open_action, llm_matches = preferred_pairs[0]
            else:
                open_action, llm_matches = rng.choice(matchable)

            llm_action = rng.choice(llm_matches)
            open_name = open_state.action_to_string(open_state.current_player(), open_action)
            llm_name = llm_game.action_to_name(llm_action)
            history.append(open_name)

            print(f"chosen move: reference {open_name} | generated {llm_name}")

            open_state.apply_action(open_action)
            llm_state = llm_apply(llm_state, llm_action)
        else:
            print(f"stopped after {OPEN_SPIEL_MAX_STEPS} steps")
    except NameError:
        print("Comparison failed: load the games first")
    except Exception as exc:
        print(f"Comparison failed: {exc}")


compare seed: 850978861
reference: openspiel
state matching: visible board + side to move
step 0
reference                                                | generated
to move: white                                           | to move: white
   a  b  c  d  e  f  g  h                                |    a  b  c  d  e  f  g  h
8 bR bN bB bQ bK bB bN bR                                | 8 bR bN bB bQ bK bB bN bR
7 bP bP bP bP bP bP bP bP                                | 7 bP bP bP bP bP bP bP bP
6 .. .. .. .. .. .. .. ..                                | 6 .. .. .. .. .. .. .. ..
5 .. .. .. .. .. .. .. ..                                | 5 .. .. .. .. .. .. .. ..
4 .. .. .. .. .. .. .. ..                                | 4 .. .. .. .. .. .. .. ..
3 .. .. .. .. .. .. .. ..                                | 3 .. .. .. .. .. .. .. ..
2 wP wP wP wP wP wP wP wP                                | 2 wP wP wP wP wP wP wP wP
1 wR wN wB wQ wK wB wN wR                                | 1 wR wN wB wQ wK wB w

- im aktuellen stand gibt es am anfang volle deckung der referenzzüge
- die generierte implementierung läuft in den ersten schritten sichtbar synchron zur referenz
- ein sofortiger grundsätzlicher regelbruch ist damit aktuell nicht erkennbar
- der erste eindruck ist deshalb eher positiv als negativ
- trotzdem ist das nur ein früher plausibilitätscheck und noch kein belastbarer korrektheitsnachweis
- interessant werden vor allem spätere stellen, an denen die deckung kleiner wird oder die sichtbaren zustände auseinanderlaufen